## Task 0: Dataset Analysis

In [1]:
import torch
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import degree

dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

node_degrees = degree(data.edge_index[0], num_nodes=data.num_nodes)
mean_degree = node_degrees.mean().item()
max_degree  = node_degrees.max().item()

print(f"Mean node degree: {mean_degree:.4f}")
print(f"Max  node degree: {max_degree:.0f}")
print(f"Feature matrix H shape: {data.x.shape}")
print(f"Label tensor dtype: {data.y.dtype}")

print("""
Explanation
-----------
Mean node degree:
  The average number of edges per node. In Cora (~3.9) this reflects a
  sparse citation graph. A low mean degree means most nodes have few
  neighbours, which affects how much information a GNN can aggregate in
  a single message-passing step.

Max node degree:
  The highest degree among all nodes (hub nodes). Very high-degree nodes
  ("hubs") can dominate aggregation and cause over-smoothing or gradient
  issues; knowing this guides choices like graph normalization or sampling.

Feature matrix H shape  (num_nodes × num_features):
  Each row is a bag-of-words feature vector for one paper. The shape tells
  us the input dimensionality of every node embedding — critical for
  designing the first GNN layer.

Label dtype (torch.int64 / Long):
  Node labels are integer class indices, confirming this is a multi-class
  node classification task. Long integers are required by PyTorch's
  CrossEntropyLoss.
""")


c:\repos\aait-lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mean node degree: 3.8981
Max  node degree: 168
Feature matrix H shape: torch.Size([2708, 1433])
Label tensor dtype: torch.int64

Explanation
-----------
Mean node degree:
  The average number of edges per node. In Cora (~3.9) this reflects a
  sparse citation graph. A low mean degree means most nodes have few
  neighbours, which affects how much information a GNN can aggregate in
  a single message-passing step.

Max node degree:
  The highest degree among all nodes (hub nodes). Very high-degree nodes
  ("hubs") can dominate aggregation and cause over-smoothing or gradient
  issues; knowing this guides choices like graph normalization or sampling.

Feature matrix H shape  (num_nodes × num_features):
  Each row is a bag-of-words feature vector for one paper. The shape tells
  us the input dimensionality of every node embedding — critical for
  designing the first GNN layer.

Label dtype (torch.int64 / Long):
  Node labels are integer class indices, confirming this is a multi-class
  nod

---

## Task 1: Implement GNN Architectures

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

IN_CHANNELS  = dataset.num_node_features
OUT_CHANNELS = dataset.num_classes
HIDDEN       = 64
DROPOUT      = 0.5

class MLP(nn.Module):
    def __init__(self, in_channels, hidden, out_channels, dropout=DROPOUT):
        super().__init__()
        self.lin1    = nn.Linear(in_channels, hidden)
        self.lin2    = nn.Linear(hidden, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index=None):   # edge_index unused
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin1(x).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        return x

class GCN(nn.Module):
    def __init__(self, in_channels, hidden, out_channels, dropout=DROPOUT):
        super().__init__()
        self.conv1   = GCNConv(in_channels, hidden)
        self.conv2   = GCNConv(hidden, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

GAT_HEADS     = 4
GAT_HEAD_DIM  = HIDDEN // GAT_HEADS   # 16

class GAT(nn.Module):
    def __init__(self, in_channels, head_dim, heads, out_channels, dropout=DROPOUT):
        super().__init__()
        self.conv1   = GATConv(in_channels, head_dim, heads=heads,
                               dropout=dropout, concat=True)
        # After concat: heads * head_dim = HIDDEN
        self.conv2   = GATConv(heads * head_dim, out_channels, heads=1,
                               dropout=dropout, concat=False)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

mlp = MLP(IN_CHANNELS, HIDDEN, OUT_CHANNELS)
gcn = GCN(IN_CHANNELS, HIDDEN, OUT_CHANNELS)
gat = GAT(IN_CHANNELS, GAT_HEAD_DIM, GAT_HEADS, OUT_CHANNELS)

print(f"MLP {count_params(mlp)}")
print(f"GCN {count_params(gcn)}")
print(f"GAT {count_params(gat)}")

print("""
Notes
-----
All three models share the same hidden dimension (64) and depth (2 layers).
• MLP  – two Linear layers, no graph information.
• GCN  – two GCNConv layers; parameter count matches MLP exactly because
         GCNConv(in, out) stores a weight matrix of the same shape as
         nn.Linear(in, out).
• GAT  – uses multi-head attention (4 heads × 16 dims).  The extra
         attention-vector parameters (att_src / att_dst per head) add only
         a small overhead (~300 extra), keeping totals comparable across
         all three models.
""")


MLP 92231
GCN 92231
GAT 92373

Notes
-----
All three models share the same hidden dimension (64) and depth (2 layers).
• MLP  – two Linear layers, no graph information.
• GCN  – two GCNConv layers; parameter count matches MLP exactly because
         GCNConv(in, out) stores a weight matrix of the same shape as
         nn.Linear(in, out).
• GAT  – uses multi-head attention (4 heads × 16 dims).  The extra
         attention-vector parameters (att_src / att_dst per head) add only
         a small overhead (~300 extra), keeping totals comparable across
         all three models.



---

## Task 2: Full-Batch Training Benchmark

In [3]:
import time

criterion = torch.nn.CrossEntropyLoss()

def train_epoch(model, optimizer):
    model.train()
    optimizer.zero_grad()
    out  = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate(model):
    model.eval()
    out  = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    val_acc  = (pred[data.val_mask]  == data.y[data.val_mask]).float().mean().item()
    test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
    return val_acc, test_acc

EPOCHS = 30
LR     = 0.01

torch.manual_seed(42)
mlp = MLP(IN_CHANNELS, HIDDEN, OUT_CHANNELS)
gcn = GCN(IN_CHANNELS, HIDDEN, OUT_CHANNELS)
gat = GAT(IN_CHANNELS, GAT_HEAD_DIM, GAT_HEADS, OUT_CHANNELS)


### MLP

In [4]:

optimizer = torch.optim.Adam(mlp.parameters(), lr=LR, weight_decay=5e-4)

t0 = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(mlp, optimizer)
    val_acc, _ = evaluate(mlp)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch: {epoch} Loss: {loss:.4f} Val Acc: {val_acc:.2%}")

elapsed_mlp = time.perf_counter() - t0
val_acc, test_acc = evaluate(mlp)

print(f"\nMLP done — val acc: {val_acc:.2%}, test acc: {test_acc:.2%}, time: {elapsed_mlp:.3f}s")


Epoch: 1 Loss: 1.9534 Val Acc: 35.60%
Epoch: 5 Loss: 1.5275 Val Acc: 57.00%
Epoch: 10 Loss: 0.7089 Val Acc: 59.00%
Epoch: 15 Loss: 0.2952 Val Acc: 58.80%
Epoch: 20 Loss: 0.2539 Val Acc: 59.60%
Epoch: 25 Loss: 0.0978 Val Acc: 58.40%
Epoch: 30 Loss: 0.0717 Val Acc: 57.40%

MLP done — val acc: 57.40%, test acc: 56.50%, time: 1.844s


### GCN


In [5]:
optimizer = torch.optim.Adam(gcn.parameters(), lr=LR, weight_decay=5e-4)

t0 = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(gcn, optimizer)
    val_acc, _ = evaluate(gcn)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch: {epoch} Loss: {loss:.4f} Val Acc: {val_acc:.2%}")

elapsed_gcn = time.perf_counter() - t0
val_acc, test_acc = evaluate(gcn)

print(f"\nGCN done — val acc: {val_acc:.2%}, test acc: {test_acc:.2%}, time: {elapsed_gcn:.3f}s")


Epoch: 1 Loss: 1.9456 Val Acc: 64.20%
Epoch: 5 Loss: 1.0020 Val Acc: 78.40%
Epoch: 10 Loss: 0.2846 Val Acc: 78.80%
Epoch: 15 Loss: 0.0858 Val Acc: 78.60%
Epoch: 20 Loss: 0.0526 Val Acc: 77.80%
Epoch: 25 Loss: 0.0436 Val Acc: 76.40%
Epoch: 30 Loss: 0.0178 Val Acc: 78.20%

GCN done — val acc: 78.20%, test acc: 80.20%, time: 2.064s


### GAT

In [6]:
optimizer = torch.optim.Adam(gat.parameters(), lr=LR, weight_decay=5e-4)

t0 = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(gat, optimizer)
    val_acc, _ = evaluate(gat)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch: {epoch} Loss: {loss:.4f} Val Acc: {val_acc:.2%}")

elapsed_gat = time.perf_counter() - t0
val_acc, test_acc = evaluate(gat)
print(f"\nGAT done — val acc: {val_acc:.2%}, test acc: {test_acc:.2%}, time: {elapsed_gat:.3f}s")


Epoch: 1 Loss: 1.9404 Val Acc: 49.80%
Epoch: 5 Loss: 1.3914 Val Acc: 77.40%
Epoch: 10 Loss: 0.9157 Val Acc: 75.80%
Epoch: 15 Loss: 0.6426 Val Acc: 76.20%
Epoch: 20 Loss: 0.5364 Val Acc: 78.00%
Epoch: 25 Loss: 0.4858 Val Acc: 78.20%
Epoch: 30 Loss: 0.4217 Val Acc: 78.00%

GAT done — val acc: 78.00%, test acc: 80.40%, time: 2.268s


---

## Task 3: Neighborhood Sampling


In [ ]:
from torch_geometric.loader import NeighborLoader
from torch_sparse import SparseTensor


row, col = data.edge_index
data.adj_t = SparseTensor(row=col, col=row,
                            sparse_sizes=(data.num_nodes, data.num_nodes))

FANOUTS    = [15, 10]
BATCH_SIZE = 64

train_loader = NeighborLoader(
    data,
    num_neighbors=FANOUTS,
    batch_size=BATCH_SIZE,
    input_nodes=data.train_mask,
    shuffle=True,
)

print(f"Fanouts: {FANOUTS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Batches/epoch: {len(train_loader)}")

torch.manual_seed(42)
mlp_s = MLP(IN_CHANNELS, HIDDEN, OUT_CHANNELS)
gcn_s = GCN(IN_CHANNELS, HIDDEN, OUT_CHANNELS)
gat_s = GAT(IN_CHANNELS, GAT_HEAD_DIM, GAT_HEADS, OUT_CHANNELS)

sampled_results = {}


Fanouts       : [15, 10]
Batch size    : 64
Batches/epoch : 3


### MLP (Sampled)


In [14]:
optimizer   = torch.optim.Adam(mlp_s.parameters(), lr=LR, weight_decay=5e-4)

t0, iterations, epoch_num = time.perf_counter(), 0, 0

while time.perf_counter() - t0 < elapsed_mlp:
    epoch_num += 1
    last_loss  = 0.0
    for batch in train_loader:
        mlp_s.train()
        optimizer.zero_grad()
        out  = mlp_s(batch.x, batch.edge_index)
        loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size])
        loss.backward()
        optimizer.step()
        last_loss   = loss.item()
        iterations += 1
        if time.perf_counter() - t0 >= elapsed_mlp:
            break
    val_acc, _ = evaluate(mlp_s)
    print(f"Epoch: {epoch_num} Iteration: {iterations} Loss: {last_loss:.4f} Val Acc: {val_acc:.2%}")

val_acc, test_acc = evaluate(mlp_s)
print(f"MLP sampled done - val acc: {val_acc:.2%}, test acc: {test_acc:.2%}, iters: {iterations}")


Epoch: 1 Iteration: 3 Loss: 0.0256 Val Acc: 53.80%
Epoch: 2 Iteration: 6 Loss: 0.4053 Val Acc: 54.00%
Epoch: 3 Iteration: 9 Loss: 0.0742 Val Acc: 52.20%
Epoch: 4 Iteration: 12 Loss: 0.0966 Val Acc: 47.60%
Epoch: 5 Iteration: 15 Loss: 0.1216 Val Acc: 46.20%
Epoch: 6 Iteration: 18 Loss: 0.2205 Val Acc: 48.60%
Epoch: 7 Iteration: 21 Loss: 0.5985 Val Acc: 50.40%
Epoch: 8 Iteration: 24 Loss: 0.2611 Val Acc: 50.20%
Epoch: 9 Iteration: 27 Loss: 0.1444 Val Acc: 49.40%
Epoch: 10 Iteration: 30 Loss: 0.8009 Val Acc: 49.40%
Epoch: 11 Iteration: 33 Loss: 0.0100 Val Acc: 50.80%
Epoch: 12 Iteration: 36 Loss: 0.1710 Val Acc: 50.60%
Epoch: 13 Iteration: 39 Loss: 0.1398 Val Acc: 52.80%
Epoch: 14 Iteration: 42 Loss: 0.0269 Val Acc: 52.40%
Epoch: 15 Iteration: 45 Loss: 0.0749 Val Acc: 51.60%
Epoch: 16 Iteration: 48 Loss: 0.2456 Val Acc: 50.80%
Epoch: 17 Iteration: 51 Loss: 0.2452 Val Acc: 49.80%
Epoch: 18 Iteration: 54 Loss: 0.0850 Val Acc: 50.20%
Epoch: 19 Iteration: 57 Loss: 0.1045 Val Acc: 50.60%
Epoch

### GCN (Sampled)


In [15]:
optimizer   = torch.optim.Adam(gcn_s.parameters(), lr=LR, weight_decay=5e-4)

t0, iterations, epoch_num = time.perf_counter(), 0, 0

while time.perf_counter() - t0 < elapsed_gcn:
    epoch_num += 1
    last_loss  = 0.0
    for batch in train_loader:
        gcn_s.train()
        optimizer.zero_grad()
        out  = gcn_s(batch.x, batch.edge_index)
        loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size])
        loss.backward()
        optimizer.step()
        last_loss   = loss.item()
        iterations += 1
        if time.perf_counter() - t0 >= elapsed_gcn:
            break
    val_acc, _ = evaluate(gcn_s)
    print(f"Epoch: {epoch_num} Iteration: {iterations} Loss: {last_loss:.4f} Val Acc: {val_acc:.2%}")

val_acc, test_acc = evaluate(gcn_s)
print(f"GCN sampled done - val acc: {val_acc:.2%}, test acc: {test_acc:.2%}, iters: {iterations}")


Epoch: 1 Iteration: 3 Loss: 0.0466 Val Acc: 77.20%
Epoch: 2 Iteration: 6 Loss: 0.0338 Val Acc: 73.60%
Epoch: 3 Iteration: 9 Loss: 0.2577 Val Acc: 75.00%
Epoch: 4 Iteration: 12 Loss: 0.1430 Val Acc: 74.40%
Epoch: 5 Iteration: 15 Loss: 0.0087 Val Acc: 76.20%
Epoch: 6 Iteration: 18 Loss: 0.0474 Val Acc: 75.80%
Epoch: 7 Iteration: 21 Loss: 0.0139 Val Acc: 75.20%
Epoch: 8 Iteration: 24 Loss: 0.0084 Val Acc: 74.20%
Epoch: 9 Iteration: 27 Loss: 0.0060 Val Acc: 74.00%
Epoch: 10 Iteration: 30 Loss: 0.0214 Val Acc: 74.60%
Epoch: 11 Iteration: 33 Loss: 0.0042 Val Acc: 74.80%
Epoch: 12 Iteration: 36 Loss: 0.0539 Val Acc: 76.00%
Epoch: 13 Iteration: 39 Loss: 0.1062 Val Acc: 76.80%
Epoch: 14 Iteration: 42 Loss: 0.0189 Val Acc: 76.60%
Epoch: 15 Iteration: 45 Loss: 0.0182 Val Acc: 77.20%
Epoch: 16 Iteration: 48 Loss: 0.0049 Val Acc: 76.00%
Epoch: 17 Iteration: 51 Loss: 0.0008 Val Acc: 76.00%
Epoch: 18 Iteration: 54 Loss: 0.0472 Val Acc: 76.00%
Epoch: 19 Iteration: 57 Loss: 0.0016 Val Acc: 76.20%
Epoch

### GAT (Sampled)


In [16]:
optimizer   = torch.optim.Adam(gat_s.parameters(), lr=LR, weight_decay=5e-4)


t0, iterations, epoch_num = time.perf_counter(), 0, 0

while time.perf_counter() - t0 < elapsed_gat:
    epoch_num += 1
    last_loss  = 0.0
    for batch in train_loader:
        gat_s.train()
        optimizer.zero_grad()
        out  = gat_s(batch.x, batch.edge_index)
        loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size])
        loss.backward()
        optimizer.step()
        last_loss   = loss.item()
        iterations += 1
        if time.perf_counter() - t0 >= elapsed_gat:
            break
    val_acc, _ = evaluate(gat_s)
    print(f"Epoch: {epoch_num} Iteration: {iterations} Loss: {last_loss:.4f} Val Acc: {val_acc:.2%}")

val_acc, test_acc = evaluate(gat_s)
print(f"GAT sampled done - val acc: {val_acc:.2%}, test acc: {test_acc:.2%}, iters: {iterations}")


Epoch: 1 Iteration: 3 Loss: 0.1998 Val Acc: 80.00%
Epoch: 2 Iteration: 6 Loss: 0.3470 Val Acc: 79.40%
Epoch: 3 Iteration: 9 Loss: 0.2897 Val Acc: 79.00%
Epoch: 4 Iteration: 12 Loss: 0.3794 Val Acc: 77.80%
Epoch: 5 Iteration: 15 Loss: 0.2051 Val Acc: 78.60%
Epoch: 6 Iteration: 18 Loss: 0.2855 Val Acc: 79.20%
Epoch: 7 Iteration: 21 Loss: 0.5706 Val Acc: 79.00%
Epoch: 8 Iteration: 24 Loss: 0.3496 Val Acc: 80.20%
Epoch: 9 Iteration: 27 Loss: 0.2358 Val Acc: 80.40%
Epoch: 10 Iteration: 30 Loss: 0.3292 Val Acc: 80.60%
Epoch: 11 Iteration: 33 Loss: 0.2621 Val Acc: 81.00%
Epoch: 12 Iteration: 36 Loss: 0.0890 Val Acc: 80.20%
Epoch: 13 Iteration: 39 Loss: 0.2182 Val Acc: 77.20%
Epoch: 14 Iteration: 42 Loss: 0.4858 Val Acc: 76.20%
Epoch: 15 Iteration: 45 Loss: 0.3231 Val Acc: 76.00%
Epoch: 16 Iteration: 48 Loss: 0.6112 Val Acc: 75.60%
Epoch: 17 Iteration: 51 Loss: 0.0359 Val Acc: 76.60%
Epoch: 18 Iteration: 54 Loss: 0.2343 Val Acc: 77.00%
Epoch: 19 Iteration: 57 Loss: 0.0044 Val Acc: 77.40%
Epoch

In [ ]:

results = {
    "MLP":  {"fullbatch": evaluate(mlp)[0],   "sampled": evaluate(mlp_s)[0]},
    "GCN":  {"fullbatch": evaluate(gcn)[0],   "sampled": evaluate(gcn_s)[0]},
    "GAT":  {"fullbatch": evaluate(gat)[0],   "sampled": evaluate(gat_s)[0]},
}

print(f"{'Model':<8} {'Fullbatch Val Acc':>18} {'Sampled Val Acc':>16} {'Diff':>8}")
print("-" * 54)
for model, accs in results.items():
    diff = accs["sampled"] - accs["fullbatch"]
    print(f"{model:<8} {accs['fullbatch']:>17.2%} {accs['sampled']:>15.2%} {diff:>+8.2%}")


Model     Fullbatch Val Acc  Sampled Val Acc     Diff
------------------------------------------------------
MLP                 57.40%          51.80%   -5.60%
GCN                 78.20%          76.20%   -2.00%
GAT                 78.00%          77.00%   -1.00%
